
# Ricci Finance v13 — yfinance-only Regime Benchmark

This notebook uses **only real market data downloaded from yfinance**.

It performs:

1. yfinance download;
2. price, volume, return, and dollar-volume preparation;
3. rolling Ricci-capital graph construction;
4. rolling feature-table generation;
5. hmmlearn Viterbi decoding;
6. hmmlearn posterior decoding;
7. duration-aware HSMM decoding;
8. optional pomegranate posterior decoding;
9. regime-engine comparison;
10. consensus regime analysis;
11. HMM story animation.

There is no synthetic-data path and no CSV fallback.



## 1. Installation

Recommended:

```bash
python -m venv .venv
source .venv/bin/activate

pip install numpy pandas matplotlib scipy scikit-learn \
            yfinance networkx plotly streamlit \
            hmmlearn GraphRicciCurvature POT networkit

# Optional pomegranate comparison
pip install torch pomegranate
```


In [ ]:

from __future__ import annotations

import math
import time
import warnings
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import plotly.graph_objects as go
import yfinance as yf

from scipy.optimize import linear_sum_assignment
from scipy.special import logsumexp
from scipy.stats import multivariate_normal

from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("yfinance:", yf.__version__)


## 2. Market configuration

In [ ]:

TICKERS = [
    "NVDA",
    "AMD",
    "AVGO",
    "TSM",
    "MU",
    "MRVL",
    "AMAT",
    "LRCX",
    "KLAC",
    "ANET",
    "AAOI",
    "COHR",
    "LITE",
    "SMCI",
    "PLTR",
    "IONQ",
    "QBTS",
    "QUBT",
    "RGTI",
    "NBIS",
]

PERIOD = "5y"
INTERVAL = "1d"

ROLLING_WINDOW = 63
ROLLING_STEP = 5
MAX_FRAMES = 120

GRAPH_MODE = "knn+bridges"
K_NEIGHBORS = 3
MIN_CORR = 0.05
MAX_BRIDGES = 3
MIN_PAIR_OBS = 20

RICCI_ALPHA = 0.5
RICCI_METHOD = "OTD"
RICCI_PROC = 1

CAPITAL_ALPHA = 0.35
USE_CAPITAL_WEIGHTING = True

N_STATES = 3


## 3. Download yfinance data

In [ ]:

def _extract_yfinance_field(
    raw: pd.DataFrame,
    field: str,
    tickers: list[str],
) -> pd.DataFrame:
    if raw.empty:
        return pd.DataFrame()

    if isinstance(raw.columns, pd.MultiIndex):
        level0 = raw.columns.get_level_values(0)
        if field not in level0:
            raise KeyError(f"{field!r} was not returned by yfinance.")
        frame = raw[field].copy()
    else:
        if field not in raw.columns:
            raise KeyError(f"{field!r} was not returned by yfinance.")
        frame = raw[[field]].copy()
        frame.columns = [tickers[0]]

    frame = frame.sort_index()
    frame = frame.loc[:, ~frame.columns.duplicated()]
    frame = frame.dropna(axis=1, how="all")
    return frame


def download_yfinance_market_data(
    tickers: list[str],
    period: str = "5y",
    interval: str = "1d",
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    tickers = list(dict.fromkeys(t.strip().upper() for t in tickers if t.strip()))

    if len(tickers) < 2:
        raise ValueError("At least two tickers are required.")

    raw = yf.download(
        tickers=tickers,
        period=period,
        interval=interval,
        auto_adjust=False,
        group_by="column",
        progress=False,
        threads=True,
    )

    close = _extract_yfinance_field(raw, "Close", tickers)
    volume = _extract_yfinance_field(raw, "Volume", tickers)

    common = close.columns.intersection(volume.columns)
    close = close[common]
    volume = volume[common]

    dollar_volume = close * volume

    return close, volume, dollar_volume


close, volume, dollar_volume = download_yfinance_market_data(
    TICKERS,
    period=PERIOD,
    interval=INTERVAL,
)

print("Price shape:", close.shape)
print("Volume shape:", volume.shape)
print("Tickers returned:", list(close.columns))
display(close.tail())


## 4. Data diagnostics and returns

In [ ]:

def validate_market_data(
    close: pd.DataFrame,
    volume: pd.DataFrame,
    minimum_rows: int,
) -> list[str]:
    messages = []

    if close.empty:
        return ["Price table is empty."]

    if len(close) < minimum_rows:
        messages.append(
            f"Only {len(close)} rows are available; "
            f"at least {minimum_rows} are required."
        )

    missing_ratio = close.isna().mean()
    high_missing = missing_ratio[missing_ratio > 0.30]

    if not high_missing.empty:
        messages.append(
            "High missing-data ratio: "
            + ", ".join(high_missing.index.astype(str))
        )

    constant = [
        column
        for column in close.columns
        if close[column].dropna().nunique() <= 1
    ]

    if constant:
        messages.append(
            "Constant or unusable price series: "
            + ", ".join(constant)
        )

    zero_volume = [
        column
        for column in volume.columns
        if volume[column].fillna(0).eq(0).all()
    ]

    if zero_volume:
        messages.append(
            "All-zero volume series: "
            + ", ".join(zero_volume)
        )

    return messages


diagnostics = validate_market_data(
    close,
    volume,
    minimum_rows=ROLLING_WINDOW + 1,
)

if diagnostics:
    for message in diagnostics:
        print("WARNING:", message)
else:
    print("Data diagnostics passed.")

returns = np.log(close / close.shift(1))
returns = returns.replace([np.inf, -np.inf], np.nan)
returns = returns.dropna(how="all")

dollar_volume = dollar_volume.reindex(
    index=returns.index,
    columns=returns.columns,
)

print("Return shape:", returns.shape)
display(returns.tail())


## 5. Ricci-capital graph functions

In [ ]:

try:
    from GraphRicciCurvature.OllivierRicci import OllivierRicci
except Exception:
    OllivierRicci = None


def financial_distance_from_corr(
    corr: pd.DataFrame,
) -> pd.DataFrame:
    clipped = corr.clip(-1.0, 1.0)
    values = np.sqrt(2.0 * (1.0 - clipped))
    return pd.DataFrame(
        values,
        index=corr.index,
        columns=corr.columns,
    )


def build_graph_from_window(
    window_returns: pd.DataFrame,
    k_neighbors: int = 3,
    min_corr: float = 0.05,
    max_bridges: int = 3,
    min_pair_obs: int = 20,
) -> tuple[nx.Graph, pd.DataFrame, pd.DataFrame]:
    usable_columns = [
        column
        for column in window_returns.columns
        if window_returns[column].notna().sum() >= min_pair_obs
    ]

    clean = window_returns[usable_columns]
    corr = clean.corr(min_periods=min_pair_obs)
    corr = corr.replace([np.inf, -np.inf], np.nan)

    dist = financial_distance_from_corr(corr)

    graph = nx.Graph()
    graph.add_nodes_from(window_returns.columns)

    for column in window_returns.columns:
        valid_obs = int(window_returns[column].notna().sum())
        graph.nodes[column]["valid_obs"] = valid_obs
        graph.nodes[column]["status"] = (
            "active"
            if valid_obs >= min_pair_obs
            else "waiting_for_data"
        )

    candidates = {}

    for source in corr.columns:
        nearest = (
            dist.loc[source]
            .drop(index=source, errors="ignore")
            .sort_values()
            .head(max(1, int(k_neighbors)))
        )

        for target, distance in nearest.items():
            rho = float(corr.loc[source, target])

            if not np.isfinite(rho) or rho < min_corr:
                continue

            edge_key = tuple(sorted((str(source), str(target))))
            candidates[edge_key] = (
                edge_key[0],
                edge_key[1],
                float(distance),
                rho,
            )

    for source, target, distance, rho in candidates.values():
        overlap = int(
            window_returns[[source, target]]
            .dropna()
            .shape[0]
        )

        confidence = min(
            1.0,
            overlap / max(min_pair_obs * 3, 1),
        )

        graph.add_edge(
            source,
            target,
            weight=distance,
            distance=distance,
            correlation=rho,
            overlap_n=overlap,
            confidence=confidence,
            bridge=False,
        )

    existing = {
        tuple(sorted(edge))
        for edge in graph.edges()
    }

    extras = []

    for index, source in enumerate(corr.columns):
        for target in corr.columns[index + 1 :]:
            edge_key = tuple(sorted((source, target)))

            if edge_key in existing:
                continue

            rho = float(corr.loc[source, target])

            if np.isfinite(rho) and rho > 0:
                distance = float(
                    np.sqrt(
                        2.0
                        * (
                            1.0
                            - np.clip(rho, -1.0, 1.0)
                        )
                    )
                )

                extras.append(
                    (
                        source,
                        target,
                        distance,
                        rho,
                    )
                )

    for source, target, distance, rho in sorted(
        extras,
        key=lambda item: item[2],
    )[: int(max_bridges)]:
        overlap = int(
            window_returns[[source, target]]
            .dropna()
            .shape[0]
        )

        graph.add_edge(
            source,
            target,
            weight=distance,
            distance=distance,
            correlation=rho,
            overlap_n=overlap,
            confidence=min(1.0, overlap / 60.0),
            bridge=True,
        )

    return graph, corr, dist


def attach_capital_attributes(
    graph: nx.Graph,
    dollar_volume_window: pd.DataFrame,
    capital_alpha: float = 0.35,
    use_capital_weighting: bool = True,
) -> nx.Graph:
    result = graph.copy()

    mass = (
        dollar_volume_window
        .reindex(columns=list(result.nodes()))
        .replace([np.inf, -np.inf], np.nan)
        .median(axis=0, skipna=True)
        .fillna(0.0)
        .clip(lower=0.0)
    )

    total_mass = float(mass.sum())

    for node in result.nodes():
        node_mass = float(mass.get(node, 0.0))

        result.nodes[node]["capital_mass"] = node_mass
        result.nodes[node]["capital_share"] = (
            node_mass / total_mass
            if total_mass > 0
            else 0.0
        )

    positive = mass[mass > 0]
    scale = (
        float(np.median(positive))
        if len(positive)
        else 1.0
    )
    log_mass = np.log1p(
        mass / max(scale, 1e-12)
    )

    for source, target, edge_data in result.edges(data=True):
        raw_distance = float(
            edge_data.get(
                "distance",
                edge_data.get("weight", 1.0),
            )
        )

        source_mass = float(mass.get(source, 0.0))
        target_mass = float(mass.get(target, 0.0))

        source_log_mass = float(
            log_mass.get(source, 0.0)
        )
        target_log_mass = float(
            log_mass.get(target, 0.0)
        )

        capital_similarity = float(
            np.exp(
                -abs(
                    source_log_mass
                    - target_log_mass
                )
            )
        )

        correlation = float(
            edge_data.get("correlation", 0.0)
        )

        edge_capital_flow = float(
            np.sqrt(
                max(source_mass, 0.0)
                * max(target_mass, 0.0)
            )
            * max(correlation, 0.0)
        )

        effective_distance = raw_distance

        if use_capital_weighting:
            effective_distance = (
                raw_distance
                / (
                    1.0
                    + float(capital_alpha)
                    * capital_similarity
                )
            )

        edge_data["raw_distance"] = raw_distance
        edge_data["capital_similarity"] = capital_similarity
        edge_data["edge_capital_flow"] = edge_capital_flow
        edge_data["distance"] = effective_distance
        edge_data["weight"] = effective_distance

    return result


def compute_ricci_curvature(
    graph: nx.Graph,
    alpha: float = 0.5,
    method: str = "OTD",
    proc: int = 1,
) -> nx.Graph:
    result = graph.copy()

    if result.number_of_edges() == 0:
        return result

    if OllivierRicci is not None:
        try:
            engine = OllivierRicci(
                result,
                alpha=alpha,
                method=method,
                weight="weight",
                proc=proc,
                verbose="ERROR",
            )

            engine.compute_ricci_curvature()
            return engine.G.copy()

        except Exception as exc:
            print(
                "GraphRicciCurvature failed; "
                "using fallback curvature:",
                exc,
            )

    for source, target in result.edges():
        common_neighbors = len(
            list(
                nx.common_neighbors(
                    result,
                    source,
                    target,
                )
            )
        )

        degree_sum = max(
            1,
            result.degree(source)
            + result.degree(target)
            - 2,
        )

        result[source][target]["ricciCurvature"] = float(
            2.0
            * common_neighbors
            / degree_sum
            - 0.5
        )

    return result


## 6. Build rolling Ricci-capital frames

In [ ]:

def component_entropy(
    component_sizes: list[int],
) -> float:
    total = sum(component_sizes)

    if total <= 0:
        return 0.0

    probabilities = (
        np.asarray(component_sizes, dtype=float)
        / total
    )

    return float(
        -np.sum(
            probabilities[probabilities > 0]
            * np.log(
                probabilities[probabilities > 0]
            )
        )
    )


def edge_jaccard(
    graph_a: nx.Graph,
    graph_b: nx.Graph,
) -> float:
    edges_a = {
        frozenset(edge)
        for edge in graph_a.edges()
    }

    edges_b = {
        frozenset(edge)
        for edge in graph_b.edges()
    }

    union = edges_a | edges_b

    return (
        1.0
        if not union
        else len(edges_a & edges_b)
        / len(union)
    )


def graph_feature_row(
    graph: nx.Graph,
    date: pd.Timestamp,
) -> dict[str, float | int | str]:
    curvatures = np.asarray(
        [
            float(
                edge_data.get(
                    "ricciCurvature",
                    0.0,
                )
            )
            for _, _, edge_data
            in graph.edges(data=True)
        ],
        dtype=float,
    )

    components = list(
        nx.connected_components(graph)
    )
    component_sizes = [
        len(component)
        for component in components
    ]

    node_capital = np.asarray(
        [
            float(
                node_data.get(
                    "capital_mass",
                    0.0,
                )
            )
            for _, node_data
            in graph.nodes(data=True)
        ],
        dtype=float,
    )

    edge_capital = np.asarray(
        [
            float(
                edge_data.get(
                    "edge_capital_flow",
                    0.0,
                )
            )
            for _, _, edge_data
            in graph.edges(data=True)
        ],
        dtype=float,
    )

    total_node_capital = float(
        node_capital.sum()
    )

    return {
        "date": str(pd.Timestamp(date).date()),
        "avg_ricci": (
            float(curvatures.mean())
            if len(curvatures)
            else 0.0
        ),
        "ricci_std": (
            float(curvatures.std())
            if len(curvatures)
            else 0.0
        ),
        "ricci_min": (
            float(curvatures.min())
            if len(curvatures)
            else 0.0
        ),
        "ricci_max": (
            float(curvatures.max())
            if len(curvatures)
            else 0.0
        ),
        "negative_edge_ratio": (
            float(
                np.mean(curvatures < 0)
            )
            if len(curvatures)
            else 0.0
        ),
        "clusters": len(components),
        "largest_component": max(
            component_sizes,
            default=0,
        ),
        "largest_component_ratio": (
            max(component_sizes, default=0)
            / max(graph.number_of_nodes(), 1)
        ),
        "nodes": graph.number_of_nodes(),
        "edges": graph.number_of_edges(),
        "density": (
            float(nx.density(graph))
            if graph.number_of_nodes() > 1
            else 0.0
        ),
        "component_entropy": component_entropy(
            component_sizes
        ),
        "total_node_capital": (
            total_node_capital
        ),
        "total_edge_capital_flow": (
            float(edge_capital.sum())
            if len(edge_capital)
            else 0.0
        ),
        "avg_edge_capital_flow": (
            float(edge_capital.mean())
            if len(edge_capital)
            else 0.0
        ),
        "max_node_capital_share": (
            float(
                node_capital.max()
                / total_node_capital
            )
            if total_node_capital > 0
            else 0.0
        ),
    }


rolling_starts = list(
    range(
        0,
        len(returns) - ROLLING_WINDOW + 1,
        ROLLING_STEP,
    )
)

if len(rolling_starts) > MAX_FRAMES:
    selected_indices = np.linspace(
        0,
        len(rolling_starts) - 1,
        MAX_FRAMES,
    ).astype(int)

    rolling_starts = [
        rolling_starts[index]
        for index in selected_indices
    ]

frames = []
feature_rows = []

for frame_index, start in enumerate(rolling_starts):
    return_window = returns.iloc[
        start : start + ROLLING_WINDOW
    ]

    graph, corr, dist = build_graph_from_window(
        return_window,
        k_neighbors=K_NEIGHBORS,
        min_corr=MIN_CORR,
        max_bridges=MAX_BRIDGES,
        min_pair_obs=MIN_PAIR_OBS,
    )

    dollar_volume_window = (
        dollar_volume
        .reindex(
            index=return_window.index,
            columns=return_window.columns,
        )
    )

    graph = attach_capital_attributes(
        graph,
        dollar_volume_window,
        capital_alpha=CAPITAL_ALPHA,
        use_capital_weighting=USE_CAPITAL_WEIGHTING,
    )

    graph = compute_ricci_curvature(
        graph,
        alpha=RICCI_ALPHA,
        method=RICCI_METHOD,
        proc=RICCI_PROC,
    )

    end_date = return_window.index[-1]

    frames.append(
        {
            "index": frame_index,
            "start": start,
            "date": end_date,
            "graph": graph,
            "corr": corr,
            "dist": dist,
        }
    )

    feature_rows.append(
        graph_feature_row(
            graph,
            end_date,
        )
    )

feature_df = pd.DataFrame(feature_rows)

feature_df["edge_stability"] = np.nan

for index in range(1, len(frames)):
    feature_df.loc[
        index,
        "edge_stability",
    ] = edge_jaccard(
        frames[index - 1]["graph"],
        frames[index]["graph"],
    )

feature_df["edge_stability"] = (
    feature_df["edge_stability"]
    .fillna(1.0)
)

display(feature_df.tail())
print("Frames:", len(frames))


## 7. Regime feature matrix

In [ ]:

REGIME_FEATURES = [
    "avg_ricci",
    "ricci_std",
    "ricci_min",
    "negative_edge_ratio",
    "clusters",
    "largest_component_ratio",
    "edges",
    "density",
    "component_entropy",
    "edge_stability",
    "total_edge_capital_flow",
    "max_node_capital_share",
]

available_features = [
    feature
    for feature in REGIME_FEATURES
    if feature in feature_df.columns
]

X_frame = (
    feature_df[available_features]
    .replace([np.inf, -np.inf], np.nan)
    .interpolate(limit_direction="both")
    .fillna(0.0)
)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_frame)

print("Regime matrix:", X_scaled.shape)
print("Features:", available_features)


## 8. hmmlearn Viterbi and posterior decoding

In [ ]:

from hmmlearn.hmm import GaussianHMM

hmm_model = GaussianHMM(
    n_components=N_STATES,
    covariance_type="diag",
    min_covar=1e-4,
    n_iter=500,
    random_state=RANDOM_STATE,
)

hmm_model.fit(X_scaled)

viterbi_states = hmm_model.predict(
    X_scaled
)

posterior_probabilities = (
    hmm_model.predict_proba(
        X_scaled
    )
)

posterior_states = (
    posterior_probabilities
    .argmax(axis=1)
)

decoder_agreement = float(
    np.mean(
        viterbi_states
        == posterior_states
    )
)

print(
    "Viterbi/posterior agreement:",
    f"{decoder_agreement:.1%}",
)

feature_df["hmm_viterbi_state"] = (
    viterbi_states
)

feature_df["hmm_posterior_state"] = (
    posterior_states
)

feature_df["hmm_state_probability"] = (
    posterior_probabilities[
        np.arange(len(posterior_states)),
        posterior_states,
    ]
)

for state in range(N_STATES):
    feature_df[
        f"hmm_probability_{state}"
    ] = posterior_probabilities[:, state]

display(
    feature_df[
        [
            "date",
            "hmm_viterbi_state",
            "hmm_posterior_state",
            "hmm_state_probability",
        ]
    ].tail()
)


## 9. Duration-aware HSMM

In [ ]:

class ExplicitDurationGaussianHSMM:
    def __init__(
        self,
        n_states: int = 3,
        min_duration: int = 3,
        max_duration: int = 40,
        random_state: int = 42,
    ):
        self.n_states = int(n_states)
        self.min_duration = int(min_duration)
        self.max_duration = int(max_duration)
        self.random_state = int(random_state)

    @staticmethod
    def _run_lengths(
        states: np.ndarray,
    ) -> dict[int, list[int]]:
        durations = {}

        if len(states) == 0:
            return durations

        start = 0

        while start < len(states):
            end = start + 1

            while (
                end < len(states)
                and states[end] == states[start]
            ):
                end += 1

            state = int(states[start])
            durations.setdefault(
                state,
                [],
            ).append(end - start)

            start = end

        return durations

    def fit(
        self,
        X: np.ndarray,
    ):
        base_model = GaussianHMM(
            n_components=self.n_states,
            covariance_type="diag",
            min_covar=1e-4,
            n_iter=500,
            random_state=self.random_state,
        )

        base_model.fit(X)
        base_states = base_model.predict(X)

        self.means_ = np.asarray(
            base_model.means_,
            dtype=float,
        )

        self.covars_ = np.asarray(
            base_model.covars_,
            dtype=float,
        )

        self.startprob_ = np.clip(
            base_model.startprob_,
            1e-12,
            1.0,
        )

        self.transmat_ = np.clip(
            base_model.transmat_,
            1e-12,
            1.0,
        )

        durations = self._run_lengths(
            base_states
        )

        self.duration_lambda_ = np.asarray(
            [
                max(
                    self.min_duration,
                    np.mean(
                        durations.get(
                            state,
                            [8],
                        )
                    ),
                )
                for state in range(
                    self.n_states
                )
            ],
            dtype=float,
        )

        return self

    def _emission_log_probability(
        self,
        X: np.ndarray,
    ) -> np.ndarray:
        output = np.empty(
            (
                len(X),
                self.n_states,
            ),
            dtype=float,
        )

        for state in range(
            self.n_states
        ):
            covariance = np.diag(
                np.maximum(
                    self.covars_[state],
                    1e-5,
                )
            )

            output[:, state] = (
                multivariate_normal.logpdf(
                    X,
                    mean=self.means_[state],
                    cov=covariance,
                    allow_singular=True,
                )
            )

        return output

    def _duration_log_probability(
        self,
        state: int,
        duration: int,
    ) -> float:
        if (
            duration < self.min_duration
            or duration > self.max_duration
        ):
            return -np.inf

        shifted_lambda = max(
            self.duration_lambda_[state]
            - self.min_duration,
            1e-6,
        )

        shifted_duration = (
            duration
            - self.min_duration
        )

        return (
            shifted_duration
            * math.log(
                shifted_lambda
            )
            - shifted_lambda
            - math.lgamma(
                shifted_duration + 1
            )
        )

    def predict(
        self,
        X: np.ndarray,
    ) -> np.ndarray:
        X = np.asarray(
            X,
            dtype=float,
        )

        sequence_length = len(X)

        emission = (
            self._emission_log_probability(
                X
            )
        )

        cumulative = np.vstack(
            [
                np.zeros(
                    self.n_states
                ),
                np.cumsum(
                    emission,
                    axis=0,
                ),
            ]
        )

        score = np.full(
            (
                sequence_length + 1,
                self.n_states,
            ),
            -np.inf,
        )

        back_state = np.full(
            (
                sequence_length + 1,
                self.n_states,
            ),
            -1,
            dtype=int,
        )

        back_duration = np.zeros(
            (
                sequence_length + 1,
                self.n_states,
            ),
            dtype=int,
        )

        log_start = np.log(
            self.startprob_
        )

        log_transition = np.log(
            self.transmat_
        )

        for end in range(
            1,
            sequence_length + 1,
        ):
            maximum_duration = min(
                self.max_duration,
                end,
            )

            for state in range(
                self.n_states
            ):
                for duration in range(
                    self.min_duration,
                    maximum_duration + 1,
                ):
                    begin = end - duration

                    segment_score = (
                        cumulative[end, state]
                        - cumulative[begin, state]
                    )

                    duration_score = (
                        self._duration_log_probability(
                            state,
                            duration,
                        )
                    )

                    if begin == 0:
                        candidate = (
                            log_start[state]
                            + segment_score
                            + duration_score
                        )

                        previous_state = -1

                    else:
                        previous_scores = (
                            score[begin]
                            + log_transition[:, state]
                        )

                        previous_scores[state] = (
                            -np.inf
                        )

                        previous_state = int(
                            np.argmax(
                                previous_scores
                            )
                        )

                        candidate = (
                            previous_scores[
                                previous_state
                            ]
                            + segment_score
                            + duration_score
                        )

                    if (
                        candidate
                        > score[end, state]
                    ):
                        score[end, state] = (
                            candidate
                        )

                        back_state[
                            end,
                            state,
                        ] = previous_state

                        back_duration[
                            end,
                            state,
                        ] = duration

        states = np.empty(
            sequence_length,
            dtype=int,
        )

        end = sequence_length
        state = int(
            np.argmax(
                score[
                    sequence_length
                ]
            )
        )

        while end > 0:
            duration = int(
                back_duration[
                    end,
                    state,
                ]
            )

            if duration <= 0:
                duration = min(
                    end,
                    self.min_duration,
                )

            begin = max(
                0,
                end - duration,
            )

            states[begin:end] = state

            state = int(
                back_state[
                    end,
                    state,
                ]
            )

            end = begin

            if state < 0 and end > 0:
                state = int(
                    np.argmax(
                        score[end]
                    )
                )

        return states


hsmm_model = ExplicitDurationGaussianHSMM(
    n_states=N_STATES,
    min_duration=3,
    max_duration=40,
    random_state=RANDOM_STATE,
).fit(X_scaled)

hsmm_states = hsmm_model.predict(
    X_scaled
)

feature_df["hsmm_state"] = (
    hsmm_states
)

print(
    "HSMM state counts:",
    np.bincount(hsmm_states),
)


## 10. Optional pomegranate posterior engine

In [ ]:

pomegranate_available = False
pomegranate_message = ""
pomegranate_probabilities = None
pomegranate_states = None

try:
    import torch
    from pomegranate.hmm import DenseHMM
    from pomegranate.distributions import Normal

    X_tensor = torch.tensor(
        X_scaled[None, :, :],
        dtype=torch.float32,
    )

    distributions = [
        Normal()
        for _ in range(N_STATES)
    ]

    pomegranate_model = DenseHMM(
        distributions,
        max_iter=250,
        tol=1e-4,
        verbose=False,
    )

    pomegranate_model.fit(
        X_tensor
    )

    pomegranate_probabilities = (
        pomegranate_model
        .predict_proba(
            X_tensor
        )[0]
        .detach()
        .cpu()
        .numpy()
    )

    pomegranate_states = (
        pomegranate_probabilities
        .argmax(axis=1)
    )

    feature_df[
        "pomegranate_state"
    ] = pomegranate_states

    pomegranate_available = True

except Exception as exc:
    pomegranate_message = str(exc)
    feature_df[
        "pomegranate_state"
    ] = -1

print(
    "pomegranate available:",
    pomegranate_available,
)

if pomegranate_message:
    print(
        "pomegranate message:",
        pomegranate_message,
    )


## 11. Decoder and engine comparison

In [ ]:

def run_lengths(
    states: np.ndarray,
) -> np.ndarray:
    states = np.asarray(
        states,
        dtype=int,
    )

    if len(states) == 0:
        return np.asarray(
            [],
            dtype=int,
        )

    boundaries = np.flatnonzero(
        np.r_[
            True,
            states[1:]
            != states[:-1],
            True,
        ]
    )

    return np.diff(boundaries)


def switch_rate(
    states: np.ndarray,
) -> float:
    states = np.asarray(
        states,
        dtype=int,
    )

    return (
        float(
            np.mean(
                states[1:]
                != states[:-1]
            )
        )
        if len(states) > 1
        else 0.0
    )


def engine_metrics(
    name: str,
    states: np.ndarray,
) -> dict[str, float | str]:
    lengths = run_lengths(states)

    return {
        "engine": name,
        "switch_rate": switch_rate(states),
        "median_run_length": (
            float(np.median(lengths))
            if len(lengths)
            else np.nan
        ),
        "mean_run_length": (
            float(np.mean(lengths))
            if len(lengths)
            else np.nan
        ),
        "short_run_ratio_le_2": (
            float(np.mean(lengths <= 2))
            if len(lengths)
            else np.nan
        ),
    }


comparison_rows = [
    engine_metrics(
        "hmmlearn Viterbi",
        viterbi_states,
    ),
    engine_metrics(
        "hmmlearn posterior",
        posterior_states,
    ),
    engine_metrics(
        "duration-aware HSMM",
        hsmm_states,
    ),
]

if pomegranate_available:
    comparison_rows.append(
        engine_metrics(
            "pomegranate posterior",
            pomegranate_states,
        )
    )

engine_comparison = pd.DataFrame(
    comparison_rows
)

display(engine_comparison)


In [ ]:

fig, axes = plt.subplots(
    3 + int(pomegranate_available),
    1,
    figsize=(
        15,
        2.4 * (
            3
            + int(
                pomegranate_available
            )
        ),
    ),
    sharex=True,
)

if not isinstance(
    axes,
    np.ndarray,
):
    axes = np.asarray([axes])

sequences = [
    (
        "hmmlearn Viterbi",
        viterbi_states,
    ),
    (
        "hmmlearn posterior",
        posterior_states,
    ),
    (
        "duration-aware HSMM",
        hsmm_states,
    ),
]

if pomegranate_available:
    sequences.append(
        (
            "pomegranate posterior",
            pomegranate_states,
        )
    )

dates = pd.to_datetime(
    feature_df["date"]
)

for axis, (
    name,
    states,
) in zip(
    axes,
    sequences,
):
    axis.step(
        dates,
        states,
        where="post",
    )

    axis.set_title(name)
    axis.set_yticks(
        list(
            range(
                N_STATES
            )
        )
    )

plt.tight_layout()
plt.show()


## 12. Consensus regime

In [ ]:

engine_state_matrix = pd.DataFrame(
    {
        "hmm_viterbi": viterbi_states,
        "hmm_posterior": posterior_states,
        "hsmm": hsmm_states,
    }
)

if pomegranate_available:
    engine_state_matrix[
        "pomegranate"
    ] = pomegranate_states


def consensus_for_row(
    row: pd.Series,
) -> tuple[int, float]:
    valid = [
        int(value)
        for value in row
        if int(value) >= 0
    ]

    if not valid:
        return -1, np.nan

    counts = pd.Series(
        valid
    ).value_counts()

    state = int(
        counts.index[0]
    )

    agreement = float(
        counts.iloc[0]
        / len(valid)
    )

    return state, agreement


consensus_values = (
    engine_state_matrix
    .apply(
        consensus_for_row,
        axis=1,
    )
)

feature_df[
    "consensus_state"
] = [
    value[0]
    for value in consensus_values
]

feature_df[
    "consensus_agreement"
] = [
    value[1]
    for value in consensus_values
]

feature_df[
    "decoder_disagreement"
] = (
    viterbi_states
    != posterior_states
)

display(
    feature_df[
        [
            "date",
            "hmm_viterbi_state",
            "hmm_posterior_state",
            "hsmm_state",
            "pomegranate_state",
            "consensus_state",
            "consensus_agreement",
            "decoder_disagreement",
        ]
    ].tail(20)
)


## 13. State interpretation

In [ ]:

def build_regime_labels(
    features: pd.DataFrame,
    states: np.ndarray,
) -> dict[int, str]:
    table = features.copy()
    table["state"] = states

    summary = (
        table
        .groupby("state")
        .agg(
            avg_ricci=(
                "avg_ricci",
                "mean",
            ),
            density=(
                "density",
                "mean",
            ),
            largest_component_ratio=(
                "largest_component_ratio",
                "mean",
            ),
            negative_edge_ratio=(
                "negative_edge_ratio",
                "mean",
            ),
        )
    )

    ordered = list(
        summary.sort_values(
            [
                "avg_ricci",
                "largest_component_ratio",
                "density",
            ]
        ).index
    )

    standard_names = [
        "stress / fragmentation",
        "transition / rotation",
        "coherent risk-on",
    ]

    labels = {}

    for index, state in enumerate(
        ordered
    ):
        labels[int(state)] = (
            standard_names[index]
            if index
            < len(standard_names)
            else f"regime_{index}"
        )

    return labels


regime_labels = build_regime_labels(
    feature_df,
    viterbi_states,
)

feature_df["regime_name"] = (
    feature_df[
        "hmm_viterbi_state"
    ].map(regime_labels)
)

print(regime_labels)

display(
    feature_df[
        [
            "date",
            "hmm_viterbi_state",
            "regime_name",
            "hmm_state_probability",
            "consensus_agreement",
        ]
    ].tail()
)


## 14. Dynamic 3-D Ricci-capital animation

In [ ]:

def build_union_graph(
    frames: list[dict],
) -> nx.Graph:
    union = nx.Graph()

    for frame in frames:
        graph = frame["graph"]

        union.add_nodes_from(
            graph.nodes()
        )

        for (
            source,
            target,
            edge_data,
        ) in graph.edges(
            data=True
        ):
            weight = float(
                edge_data.get(
                    "weight",
                    1.0,
                )
            )

            if union.has_edge(
                source,
                target,
            ):
                union[source][target][
                    "weight"
                ] = min(
                    union[source][target][
                        "weight"
                    ],
                    weight,
                )
            else:
                union.add_edge(
                    source,
                    target,
                    weight=weight,
                )

    return union


union_graph = build_union_graph(
    frames
)

positions = nx.spring_layout(
    union_graph,
    dim=3,
    seed=RANDOM_STATE,
    weight="weight",
    iterations=500,
)

positions = {
    node: tuple(
        float(value)
        for value in coordinates
    )
    for node, coordinates
    in positions.items()
}


def node_z_value(
    graph: nx.Graph,
    node: str,
) -> float:
    curvatures = [
        float(
            graph[node][neighbor]
            .get(
                "ricciCurvature",
                0.0,
            )
        )
        for neighbor in graph.neighbors(
            node
        )
    ]

    return (
        float(
            -np.mean(
                curvatures
            )
        )
        if curvatures
        else 0.0
    )


def ricci_color(
    curvature: float,
) -> str:
    curvature = float(
        np.clip(
            curvature,
            -0.6,
            0.6,
        )
    )

    normalized = (
        curvature + 0.6
    ) / 1.2

    red = int(
        255
        * (
            1.0
            - normalized
        )
    )

    blue = int(
        255
        * normalized
    )

    return f"rgb({red},80,{blue})"


def graph_frame_traces(
    frame_index: int,
    show_edge_labels: bool = True,
    top_edge_labels: int = 20,
):
    graph = frames[
        frame_index
    ]["graph"]

    traces = []

    edge_records = []

    for (
        source,
        target,
        edge_data,
    ) in graph.edges(
        data=True
    ):
        source_x, source_y, _ = (
            positions[source]
        )
        target_x, target_y, _ = (
            positions[target]
        )

        source_z = node_z_value(
            graph,
            source,
        )
        target_z = node_z_value(
            graph,
            target,
        )

        curvature = float(
            edge_data.get(
                "ricciCurvature",
                0.0,
            )
        )

        flow = float(
            edge_data.get(
                "edge_capital_flow",
                0.0,
            )
        )

        correlation = float(
            edge_data.get(
                "correlation",
                np.nan,
            )
        )

        distance = float(
            edge_data.get(
                "distance",
                np.nan,
            )
        )

        traces.append(
            go.Scatter3d(
                x=[
                    source_x,
                    target_x,
                ],
                y=[
                    source_y,
                    target_y,
                ],
                z=[
                    source_z,
                    target_z,
                ],
                mode="lines",
                line={
                    "width": (
                        1.0
                        + 4.0
                        * np.sqrt(
                            max(
                                flow,
                                0.0,
                            )
                            / max(
                                1.0,
                                flow,
                            )
                        )
                    ),
                    "color": ricci_color(
                        curvature
                    ),
                },
                hovertext=(
                    f"{source}–{target}<br>"
                    f"ρ={correlation:.3f}<br>"
                    f"d={distance:.3f}<br>"
                    f"κ={curvature:.3f}<br>"
                    f"flow={flow:,.0f}"
                ),
                hoverinfo="text",
                showlegend=False,
            )
        )

        edge_records.append(
            {
                "source": source,
                "target": target,
                "x": (
                    source_x
                    + target_x
                )
                / 2.0,
                "y": (
                    source_y
                    + target_y
                )
                / 2.0,
                "z": (
                    source_z
                    + target_z
                )
                / 2.0,
                "flow": flow,
                "label": (
                    f"ρ={correlation:.2f}<br>"
                    f"d={distance:.2f}<br>"
                    f"κ={curvature:.2f}"
                ),
            }
        )

    if show_edge_labels:
        selected_edges = sorted(
            edge_records,
            key=lambda item: item[
                "flow"
            ],
            reverse=True,
        )[: int(top_edge_labels)]

        traces.append(
            go.Scatter3d(
                x=[
                    edge["x"]
                    for edge in selected_edges
                ],
                y=[
                    edge["y"]
                    for edge in selected_edges
                ],
                z=[
                    edge["z"]
                    for edge in selected_edges
                ],
                mode="text",
                text=[
                    edge["label"]
                    for edge in selected_edges
                ],
                textfont={
                    "size": 9,
                },
                hoverinfo="skip",
                showlegend=False,
            )
        )

    node_x = []
    node_y = []
    node_z = []
    node_text = []
    node_size = []
    node_hover = []

    capital_values = np.asarray(
        [
            float(
                graph.nodes[node]
                .get(
                    "capital_mass",
                    0.0,
                )
            )
            for node in graph.nodes()
        ],
        dtype=float,
    )

    capital_scale = (
        np.percentile(
            capital_values[
                capital_values > 0
            ],
            90,
        )
        if np.any(
            capital_values > 0
        )
        else 1.0
    )

    for node in sorted(
        graph.nodes()
    ):
        x, y, _ = positions[node]
        z = node_z_value(
            graph,
            node,
        )

        capital_mass = float(
            graph.nodes[node]
            .get(
                "capital_mass",
                0.0,
            )
        )

        node_x.append(x)
        node_y.append(y)
        node_z.append(z)
        node_text.append(node)
        node_size.append(
            9.0
            + 20.0
            * np.sqrt(
                max(
                    capital_mass,
                    0.0,
                )
                / max(
                    capital_scale,
                    1e-12,
                )
            )
        )

        node_hover.append(
            f"Ticker: {node}<br>"
            f"capital={capital_mass:,.0f}<br>"
            f"Ricci stress z={z:.3f}"
        )

    traces.append(
        go.Scatter3d(
            x=node_x,
            y=node_y,
            z=node_z,
            mode="markers+text",
            text=node_text,
            textposition="top center",
            marker={
                "size": node_size,
                "line": {
                    "width": 1,
                    "color": "#222",
                },
            },
            hovertext=node_hover,
            hoverinfo="text",
            showlegend=False,
        )
    )

    return traces


initial_traces = graph_frame_traces(
    0,
    show_edge_labels=True,
    top_edge_labels=20,
)

figure = go.Figure(
    data=initial_traces
)

plotly_frames = []

for frame_index in range(
    len(frames)
):
    probability = float(
        feature_df.loc[
            frame_index,
            "hmm_state_probability",
        ]
    )

    regime_name = str(
        feature_df.loc[
            frame_index,
            "regime_name",
        ]
    )

    plotly_frames.append(
        go.Frame(
            name=str(
                frame_index
            ),
            data=graph_frame_traces(
                frame_index,
                show_edge_labels=True,
                top_edge_labels=20,
            ),
            layout=go.Layout(
                title={
                    "text": (
                        "Ricci Finance v13 "
                        "yfinance-only<br>"
                        f"<sup>Frame "
                        f"{frame_index + 1} · "
                        f"{feature_df.loc[frame_index, 'date']} · "
                        f"{regime_name} · "
                        f"posterior "
                        f"{probability:.1%}"
                        "</sup>"
                    )
                }
            ),
        )
    )

figure.frames = plotly_frames

slider_steps = [
    {
        "method": "animate",
        "label": str(
            frame_index + 1
        ),
        "args": [
            [
                str(
                    frame_index
                )
            ],
            {
                "mode": "immediate",
                "frame": {
                    "duration": 700,
                    "redraw": True,
                },
                "transition": {
                    "duration": 0,
                },
            },
        ],
    }
    for frame_index in range(
        len(frames)
    )
]

figure.update_layout(
    title=(
        "Ricci Finance v13 "
        "yfinance-only"
    ),
    height=850,
    showlegend=False,
    scene={
        "xaxis": {
            "visible": False,
        },
        "yaxis": {
            "visible": False,
        },
        "zaxis": {
            "title": "Ricci stress",
        },
        "camera": {
            "eye": {
                "x": 1.8,
                "y": 1.8,
                "z": 1.15,
            }
        },
        "aspectmode": "cube",
    },
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {
                                "duration": 700,
                                "redraw": True,
                            },
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {
                                "duration": 0,
                            },
                            "mode": "immediate",
                        },
                    ],
                },
            ],
        }
    ],
    sliders=[
        {
            "active": 0,
            "currentvalue": {
                "prefix": "Frame ",
            },
            "steps": slider_steps,
        }
    ],
)

figure.show()


## 15. Export results

In [ ]:

OUTPUT_FEATURE_CSV = (
    "v13_yfinance_ricci_regime_features.csv"
)

feature_df.to_csv(
    OUTPUT_FEATURE_CSV,
    index=False,
)

print(
    "Saved:",
    OUTPUT_FEATURE_CSV,
)



## Interpretation

- The main story path uses **hmmlearn Viterbi**.
- Posterior probabilities provide frame confidence.
- HSMM tests whether explicit duration produces more economically plausible regimes.
- pomegranate is optional and treated as posterior-only unless its installed version provides a compatible path decoder.
- Consensus disagreement is a warning that regime interpretation is model-sensitive.
- Ricci flow or Ricci curvature is not, by itself, a future-price forecast.
